In [44]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [45]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [46]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [47]:
# Add Compounds
cnames = ["Water"]
sim.AddCompound("Water")

In [48]:
one = sim.AddFlowsheetObject('Material Stream','1')
two = sim.AddFlowsheetObject('Material Stream','2')
PUMP_1 = sim.AddFlowsheetObject('Pump','PUMP-1')
E_1 = sim.AddFlowsheetObject('Energy Stream','E1')
C_1 = sim.AddFlowsheetObject('Controller Block','C-1')

In [49]:
one = one.GetAsObject()
two = two.GetAsObject()
PUMP_1 = PUMP_1.GetAsObject()
E_1 = E_1.GetAsObject()
C_1 = C_1.GetAsObject()

In [50]:
sim.ConnectObjects(one.GraphicObject , PUMP_1.GraphicObject, -1, -1)
sim.ConnectObjects(E_1.GraphicObject , PUMP_1.GraphicObject, -1, -1)
sim.ConnectObjects(PUMP_1.GraphicObject, two.GraphicObject, -1, -1)

In [51]:
one.SetOverallComposition(Array[float]([1]))
one.SetTemperature(350.0) # K
one.SetPressure(101325.0) # Pa
one.SetMassFlow(1.0) # kg/s

'1: mass flow set to 1 kg/s'

In [52]:
# property package
Steam_table = sim.CreateAndAddPropertyPackage("Steam Tables (IAPWS-IF97)")

In [53]:
# Calc Modes
PUMP_1.CalcMode = PUMP_1.CalcMode.Power
PUMP_1.Eficiencia = 70
PUMP_1.set_DeltaQ(2)

In [ ]:
# Setting up the controller block to control the pump's power requirement based on the outlet pressure of the pump. The controller will adjust the pump's power to maintain a desired outlet pressure.
from DWSIM.UnitOperations.SpecialOps.Helpers import SpecialOpObjectInfo

# --- Manipulated side (Pump power) ---
mv_info = SpecialOpObjectInfo()
mv_info.ID = PUMP_1.Name
mv_info.Name = PUMP_1.GraphicObject.Tag  
mv_info.PropertyName = "PROP_PU_3"    

C_1.set_ManipulatedObject(PUMP_1)
C_1.set_ManipulatedObjectData(mv_info)

# --- Controlled side (outlet stream pressure) ---
cv_info = SpecialOpObjectInfo()
cv_info.ID = two.Name
cv_info.Name = two.GraphicObject.Tag
cv_info.PropertyName = "PROP_MS_1"

C_1.set_ControlledObject(two)
C_1.set_ControlledObjectData(cv_info)

C_1.set_AdjustValue(2300000.0)  # target, Pa

C_1.SimultaneousAdjust = True


In [55]:
sim.AutoLayout()

In [56]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [57]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/20_Controller_Block/20 Controller Block Model.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)